# 06 Baseline Comparisons

Evaluate simpler anomaly detectors on the same frozen split as the CNN autoencoder. Thresholds are selected on validation data and then frozen for the test split.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM


In [ ]:
REPO_ROOT = Path('..').resolve()
MANIFEST_PATH = REPO_ROOT / 'reports' / 'manifests' / 'turning_split_seed42.csv'
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
FS = 50000
MAX_FREQ = 5000
PCA_COMPONENTS = 32
SEED = 42

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(MANIFEST_PATH)
manifest['target'] = (manifest['label'] == 'chatter').astype(int)
manifest.groupby(['split', 'label']).size().rename('n').reset_index()


In [ ]:
def spectral_features(signal: np.ndarray, fs: int = FS) -> dict:
    freqs, psd = welch(signal, fs=fs, nperseg=min(2048, len(signal)))
    mask = freqs <= MAX_FREQ
    freqs = freqs[mask]
    psd = psd[mask]
    total = float(np.sum(psd) + 1e-12)
    centroid = float(np.sum(freqs * psd) / total)
    bandwidth = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * psd) / total))
    dominant_frequency = float(freqs[int(np.argmax(psd))]) if len(freqs) else np.nan
    bands = [(0, 1000), (1000, 2000), (2000, 3000), (3000, 4000), (4000, 5000)]
    band_values = {f'band_{lo}_{hi}': float(np.sum(psd[(freqs >= lo) & (freqs < hi)])) for lo, hi in bands}
    return {'spectral_centroid': centroid, 'spectral_bandwidth': bandwidth, 'dominant_frequency': dominant_frequency, **band_values}

def window_features(npz_path: Path) -> dict:
    with np.load(npz_path) as data:
        features = {}
        for axis in ['X', 'Y', 'Z']:
            values = data[axis].astype(float)
            rms = float(np.sqrt(np.mean(values ** 2)))
            peak = float(np.max(np.abs(values)))
            features[f'{axis}_rms'] = rms
            features[f'{axis}_peak_abs'] = peak
            features[f'{axis}_crest_factor'] = peak / max(rms, 1e-12)
            features.update({f'{axis}_{k}': v for k, v in spectral_features(values).items()})
    return features

def image_vector(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert('RGB').resize(IMAGE_SIZE)
    return np.asarray(image, dtype='float32').reshape(-1) / 255.0


In [ ]:
feature_rows = []
image_rows = []
for _, row in manifest.iterrows():
    npz_path = REPO_ROOT / row['npz_path']
    feature_rows.append({'sample_id': row['sample_id'], **window_features(npz_path)})
    if isinstance(row.get('image_path'), str) and row['image_path']:
        image_rows.append({'sample_id': row['sample_id'], 'image_vector': image_vector(REPO_ROOT / row['image_path'])})

feature_table = manifest.merge(pd.DataFrame(feature_rows), on='sample_id', how='left')
image_table = manifest.merge(pd.DataFrame(image_rows), on='sample_id', how='inner')
print('feature rows:', len(feature_table))
print('image rows:', len(image_table))


In [ ]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {'threshold': float(thresholds[best_idx]), 'validation_f1': float(f1[best_idx])}

def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


In [ ]:
feature_columns = [c for c in feature_table.columns if c.startswith(('X_', 'Y_', 'Z_'))]
train_nominal = feature_table[(feature_table['split'] == 'train') & (feature_table['label'] == 'no_chatter')]
validation = feature_table[feature_table['split'] == 'validation']
test = feature_table[feature_table['split'] == 'test']

scaler = StandardScaler().fit(train_nominal[feature_columns])
X_train = scaler.transform(train_nominal[feature_columns])
X_val = scaler.transform(validation[feature_columns])
X_test = scaler.transform(test[feature_columns])
y_val = validation['target'].to_numpy()
y_test = test['target'].to_numpy()


In [ ]:
baseline_metrics = []
baseline_score_frames = []
baseline_thresholds = {}

ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05).fit(X_train)
iforest = IsolationForest(random_state=SEED, contamination='auto').fit(X_train)
models = {
    'one_class_svm_features': lambda x: -ocsvm.decision_function(x),
    'isolation_forest_features': lambda x: -iforest.decision_function(x),
}

for method, score_fn in models.items():
    val_scores = score_fn(X_val)
    test_scores = score_fn(X_test)
    selected = select_best_f1_threshold(y_val, val_scores)
    baseline_thresholds[method] = selected
    metrics = evaluate_at_threshold(y_test, test_scores, selected['threshold'])
    baseline_metrics.append({'dataset': 'turning', 'method': method, 'score': 'anomaly_score', 'threshold': selected['threshold'], **metrics})

    baseline_score_frames.append(pd.DataFrame({
        'dataset': 'turning',
        'method': method,
        'sample_id': validation['sample_id'].to_numpy(),
        'split': 'validation',
        'label': validation['label'].to_numpy(),
        'target': y_val,
        'score_value': val_scores,
    }))
    baseline_score_frames.append(pd.DataFrame({
        'dataset': 'turning',
        'method': method,
        'sample_id': test['sample_id'].to_numpy(),
        'split': 'test',
        'label': test['label'].to_numpy(),
        'target': y_test,
        'score_value': test_scores,
    }))


In [ ]:
image_train = image_table[(image_table['split'] == 'train') & (image_table['label'] == 'no_chatter')]
image_val = image_table[image_table['split'] == 'validation']
image_test = image_table[image_table['split'] == 'test']
X_img_train = np.stack(image_train['image_vector'].to_numpy())
X_img_val = np.stack(image_val['image_vector'].to_numpy())
X_img_test = np.stack(image_test['image_vector'].to_numpy())
y_img_val = image_val['target'].to_numpy()
y_img_test = image_test['target'].to_numpy()

pca = PCA(n_components=min(PCA_COMPONENTS, X_img_train.shape[0], X_img_train.shape[1]), random_state=SEED).fit(X_img_train)
val_recon = pca.inverse_transform(pca.transform(X_img_val))
test_recon = pca.inverse_transform(pca.transform(X_img_test))
val_scores = np.mean((X_img_val - val_recon) ** 2, axis=1)
test_scores = np.mean((X_img_test - test_recon) ** 2, axis=1)
method = 'pca_image_reconstruction'
selected = select_best_f1_threshold(y_img_val, val_scores)
baseline_thresholds[method] = selected
metrics = evaluate_at_threshold(y_img_test, test_scores, selected['threshold'])
baseline_metrics.append({'dataset': 'turning', 'method': method, 'score': 'reconstruction_mse', 'threshold': selected['threshold'], **metrics})

baseline_score_frames.append(pd.DataFrame({
    'dataset': 'turning',
    'method': method,
    'sample_id': image_val['sample_id'].to_numpy(),
    'split': 'validation',
    'label': image_val['label'].to_numpy(),
    'target': y_img_val,
    'score_value': val_scores,
}))
baseline_score_frames.append(pd.DataFrame({
    'dataset': 'turning',
    'method': method,
    'sample_id': image_test['sample_id'].to_numpy(),
    'split': 'test',
    'label': image_test['label'].to_numpy(),
    'target': y_img_test,
    'score_value': test_scores,
}))


In [ ]:
baseline_metrics = pd.DataFrame(baseline_metrics)
baseline_scores = pd.concat(baseline_score_frames, ignore_index=True)
METRICS_PATH = TABLE_DIR / 'metrics_turning_baselines.csv'
SCORES_PATH = BASELINE_DIR / 'baseline_scores_turning.csv'
THRESHOLD_PATH = THRESHOLD_DIR / 'baseline_thresholds_turning.json'

baseline_metrics.to_csv(METRICS_PATH, index=False)
baseline_scores.to_csv(SCORES_PATH, index=False)
with THRESHOLD_PATH.open('w') as f:
    json.dump(baseline_thresholds, f, indent=2)

print(f'Wrote {METRICS_PATH}')
print(f'Wrote {SCORES_PATH}')
print(f'Wrote {THRESHOLD_PATH}')
baseline_metrics
